In [ ]:
!pip install -q delta-spark pyspark

In [2]:
import delta
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("Part_8_merge") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print(delta.__version__)
print("Spark session ready")

4.2.0
Spark session ready


In [3]:
orders_data = [
(301,101,201,"2024-04-01",20,"Delivered"),
(302,102,201,"2024-04-01",35,"Delivered"),
(303,111,204,"2024-04-02",2,"Delivered"),
(304,114,208,"2024-04-02",5,"Pending"),
(305,115,204,"2024-04-03",3,"Delivered"),
(306,104,202,"2024-04-03",50,"Delivered"),
(307,105,202,"2024-04-04",18,"Cancelled"),
(308,117,206,"2024-04-04",7,"Delivered"),
(309,118,210,"2024-04-05",4,"Pending"),
(310,119,206,"2024-04-05",12,"Delivered"),
(311,120,210,"2024-04-06",6,"Delivered"),
(312,113,204,"2024-04-06",4,"Delivered"),
(313,116,208,"2024-04-07",2,"Pending"),
(314,109,205,"2024-04-07",80,"Delivered"),
(315,110,205,"2024-04-08",120,"Delivered"),
(316,106,203,"2024-04-08",60,"Cancelled"),
(317,107,209,"2024-04-09",25,"Delivered"),
(318,108,203,"2024-04-09",40,"Delivered"),
(319,112,208,"2024-04-10",2,"Pending"),
(320,101,207,"2024-04-10",15,"Delivered")
]
orders_columns = [
"order_id",
"product_id",
"supplier_id",
"order_date",
"quantity",
"order_status"
]
orders_df = spark.createDataFrame(orders_data, orders_columns)

In [ ]:
daily_orders_data = [
(321,114,204,"2024-04-11",3,"Delivered"),
(322,118,210,"2024-04-11",2,"Delivered"),
(304,114,208,"2024-04-02",5,"Delivered"),
(319,112,208,"2024-04-10",2,"Delivered"),
(323,120,210,"2024-04-12",1,"Pending")
]

daily_orders_columns = [
"order_id",
"product_id",
"supplier_id",
"order_date",
"quantity",
"order_status"
]

daily_orders_df = spark.createDataFrame(daily_orders_data, daily_orders_columns)
daily_orders_df.write.format("delta").mode("overwrite").save("/tmp/daily_orders_updates")
spark.sql(""" create table if not exists daily_orders_updates using delta
location '/tmp/daily_orders_updates' """)
spark.sql(""" select * from daily_orders_updates """).show()
#Exercise 71
orders_df.write.format("delta").mode("overwrite").save("/tmp/orders_target")


In [ ]:
#Exercise 72
target_df= spark.read.format("delta").load("/tmp/orders_target")
target_df.show()

In [14]:
#Exercise 73
daily_orders_df.createOrReplaceTempView("daily_orders_updates")

In [16]:
#Exercise 74
spark.sql(""" merge into orders_target t
using daily_orders_updates s
on t.order_id = s.order_id
when matched then update set *
when not matched then insert * """)


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [17]:
#Exercise 75
spark.sql(""" merge into orders_target t
using daily_orders_updates s
on t.order_id = s.order_id
when matched then update set
t.product_id = s.product_id,
t.supplier_id = s.supplier_id,
t.order_date = s.order_date,
t.quantity= s.quantity,
t.order_status = s.order_status
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [19]:
#Exercise 76
spark.sql(""" merge into orders_target t
using daily_orders_updates s
on t.order_id = s.order_id
when not matched then insert
(t.order_id, t.product_id, t.supplier_id, t.order_date, t.quantity, t.order_status)
values
(s.order_id,s.product_id,s.supplier_id,s.order_date,s.quantity,s.order_status)
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [20]:
#Exercise 77
spark.sql(""" select s.order_id from daily_orders_updates s
join orders_target t on s.order_id = t.order_id """).show()

+--------+
|order_id|
+--------+
|     321|
|     322|
|     304|
|     319|
|     323|
+--------+



In [22]:
#Exercise 78
spark.sql(""" select s.order_id from daily_orders_updates s
left anti join orders_target t on s.order_id = t.order_id """).show()

+--------+
|order_id|
+--------+
+--------+



In [23]:
#Exercise 79
spark.sql(""" describe history orders_target """).show()

+-------+--------------------+------+--------+---------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+---------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      6|2026-05-06 09:16:...|  NULL|    NULL|    MERGE|{predicate -> ["(...|NULL|    NULL|     NULL|          5|  Serializable|        false|{numTargetRowsCop...|        NULL|Apache-Spark/4.0....|
|      5|2026-05-06 09:12:...|  NULL|    NULL|    MERGE|{predicate -> ["(...|NULL|    NULL|     NULL|          4|  Serializable|        false|{numTargetRowsCop...|        NULL|Apache-Spark/4.0....|
|      4|2

In [ ]:
#Exercise 80
spark.sql(" select * from orders_target ").show()

MERGE is important because it performs UPSERT (update + insert) in one command.
It updates existing records and inserts new records without creating duplicates.
It is widely used in incremental ETL pipelines to maintain latest data accurately.